# Cross-camera GAP LeJEPA for cotton boll detection

This notebook **only configures and calls** functions from the `lejepa_cotton` package; no functions are defined here.

1. **Pretraining** - an empty-weight `yolov8n.yaml` backbone is trained with LeJEPA on synchronised frames from cameras 1, 2 and 4. Every scale (P3, P4, P5) is globally average pooled and projected; the prediction loss pulls each camera's embedding toward the mean of the *other* cameras of the same frame, and SIGReg keeps the embeddings isotropic Gaussian.
2. **Evaluation** - the LeJEPA backbone is compared with COCO `yolov8n.pt` weights on (a) cotton-boll detection fine-tuning and (b) a frozen-backbone GAP linear probe for plot status.
3. **Visualization** - loss curves, camera-coloured PCA of embeddings, detection curves/overlays, confusion matrices and metric bars.

## 0. Install the package

The repository is opened in VS Code as a **GitHub remote repository** (virtual workspace), so the files are not on your Mac's disk and the kernel starts in `/` (read-only). Therefore:

* the package is installed straight from the `Packaging` branch on GitHub;
* every path below is an **absolute local path** on your Mac.

`--force-reinstall --no-deps` makes pip fetch the latest commit even though the version number (`0.1.0`) has not changed. Re-run this cell after every push to `Packaging`, then **restart the kernel**.

In [1]:
%pip install --force-reinstall --no-deps "git+https://github.com/Akintanoreofe/SSL_for_Cotton-field_Assessment_Using_CV.git@Packaging#subdirectory=lejepa_cotton_gap"

  Cloning https://github.com/Akintanoreofe/SSL_for_Cotton-field_Assessment_Using_CV.git (to revision Packaging) to ./private/var/folders/93/l8t4dgh52fx3v_ng2z7jhf8h0000gn/T/pip-req-build-a1hnrjha
  Running command git clone --filter=blob:none --quiet https://github.com/Akintanoreofe/SSL_for_Cotton-field_Assessment_Using_CV.git /private/var/folders/93/l8t4dgh52fx3v_ng2z7jhf8h0000gn/T/pip-req-build-a1hnrjha
  Running command git checkout -b Packaging --track origin/Packaging
  Switched to a new branch 'Packaging'
  branch 'Packaging' set up to track 'origin/Packaging'.
  Resolved https://github.com/Akintanoreofe/SSL_for_Cotton-field_Assessment_Using_CV.git to commit 94d6bfb5cca1aa8c0f3de1b30273ef2754b3ad2d
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for lejepa-cotton: filename=lejepa_cotton-0.1.0-py3-none-any.whl size=29500 sha256=ce689e8481a8ee1450449590174a1998b6a6a87ec45687865d3e

**Alternative - local clone (recommended while developing).** Clone the branch once in a terminal, open the cloned folder in VS Code (*File > Open Folder*), and install in editable mode. Edits then apply after a kernel restart and hover docstrings work:

```bash
git clone -b Packaging https://github.com/Akintanoreofe/SSL_for_Cotton-field_Assessment_Using_CV.git ~/SSL_for_Cotton-field_Assessment_Using_CV
```
```python
%pip install -e ~/SSL_for_Cotton-field_Assessment_Using_CV/lejepa_cotton_gap --config-settings editable_mode=compat
```

## 1. Check the installation and read the docstrings
Restart the kernel after installing, then run this cell. `help(...)` prints the NumPy docstrings inside the notebook even when the editor cannot show them on hover.

In [2]:
import lejepa_cotton

print("version:", lejepa_cotton.__version__)
print("installed at:", lejepa_cotton.__file__)
help(lejepa_cotton.run_pretraining)

version: 0.1.0
installed at: /opt/anaconda3/envs/ML_DL_CV/lib/python3.12/site-packages/lejepa_cotton/__init__.py
Help on function run_pretraining in module lejepa_cotton.pipeline:

run_pretraining(cfg: 'PretrainConfig', pca_epochs: 'Optional[Iterable[int]]' = None, pca_max_batches: 'int' = 60) -> 'Tuple[Path, pd.DataFrame]'
    Pretrain, then save the checkpoint, loss history and plots.

    Parameters
    ----------
    cfg : PretrainConfig
        Pretraining configuration.
    pca_epochs : iterable of int or None, default=None
        Zero-based epochs to visualise; defaults to first, middle and last.
    pca_max_batches : int, default=60
        Batches used per PCA snapshot.

    Returns
    -------
    checkpoint : pathlib.Path
        Saved encoder checkpoint.
    history : pandas.DataFrame
        Per-epoch loss components.



In [3]:
from pathlib import Path
import os

from IPython.display import Image, display

from lejepa_cotton import (
    DetectionEvalConfig,
    PretrainConfig,
    ProbeEvalConfig,
    WeightSources,
    run_detection_evaluation,
    run_pretraining,
    run_probe_evaluation,
)

## 2. Paths
The only place directories are defined. All paths are **absolute** because the kernel does not start inside the repository.

`os.chdir(OUTPUT_ROOT)` moves the kernel out of the read-only `/`, so files Ultralytics writes relative to the working directory (e.g. the downloaded `yolov8n.pt`) land in `OUTPUT_ROOT`.

In [5]:
HOME = Path.home()

MULTI_CAMERA_ROOT = HOME / "Downloads" / "LeJEPA _pretrainining_multi_camera_boll" / "mars_multi_camera_boll"
DETECTION_DATA = HOME / "Downloads" / "detection_dataset"            # contains images/ and labels/  <- edit
PLOT_STATUS_DIR = HOME / "Downloads" / "LeJEPA _pretrainining_multi_camera_boll" / "Plot Status"             # images + annotations.json     <- edit
OUTPUT_ROOT = HOME / "lejepa_cotton_outputs"                     # writable, outside the repo

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(OUTPUT_ROOT)

print("working directory:", Path.cwd())
print("multi-camera data found:", MULTI_CAMERA_ROOT.exists())
print("detection data found:   ", DETECTION_DATA.exists())
print("plot-status data found: ", (PLOT_STATUS_DIR / "annotations.json").exists())

working directory: /Users/akintanoreofeoluwa/lejepa_cotton_outputs
multi-camera data found: True
detection data found:    True
plot-status data found:  True


## 3. Cross-camera GAP LeJEPA pretraining
`max_samples` counts synchronised frame **triplets** (one image per camera), so 16,667 triplets is about 50k images. PCA snapshots are written at the first, middle and last epoch, coloured by camera: well-mixed colours mean viewpoint-invariant embeddings.

In [ ]:
pretrain_cfg = PretrainConfig(
    image_root=MULTI_CAMERA_ROOT,
    output_dir=OUTPUT_ROOT / "pretraining",
    cameras=(1, 2, 4),        # 2nd, 3rd and 5th physical cameras (0-indexed in file names)
    max_samples=16_667,
    views_per_camera=1,
    image_size=128,
    batch_size=16,
    epochs=60,
    proj_dim=128,
    lr=1e-3,
    weight_decay=1e-4,
    lam=0.2,
    model_cfg="yolov8n.yaml",  # empty weights
)

checkpoint, history = run_pretraining(pretrain_cfg)
history.tail()

Synchronised frames for cameras (1, 2, 4): 28842 found, 16667 used (50001 images).


Epoch 1/60:  16%|█▋        | 171/1041 [00:56<05:08,  2.82it/s]

In [ ]:
display(Image(filename=str(pretrain_cfg.output_dir / "plots" / "loss_curves.png")))
print("Interactive PCA snapshots:", *sorted((pretrain_cfg.output_dir / "plots" / "pca_3d").glob("*.html")), sep="\n")

## 4. Evaluation A - cotton boll detection fine-tuning (LeJEPA vs COCO)
Both variants are fine-tuned on the **same** train/val split with identical hyper-parameters. Add `"coco_backbone"` (COCO backbone, random neck/head - the like-for-like control) or `"scratch"` to `variants` for extra baselines.

To evaluate an existing checkpoint without re-running pretraining, replace `checkpoint` with its path, e.g. `OUTPUT_ROOT / "pretraining" / "gap_lejepa_yolov8n.pth"`.

In [ ]:
weights = WeightSources(lejepa_checkpoint=checkpoint, model_cfg="yolov8n.yaml", coco_weights="yolov8n.pt")

detection_cfg = DetectionEvalConfig(
    source_dir=DETECTION_DATA,
    output_dir=OUTPUT_ROOT / "detection",
    weights=weights,
    class_names=("cotton_boll",),
    variants=("lejepa", "coco"),
    image_size=256,
    batch_size=16,
    epochs=40,
    lr0=0.002,
    device="cpu",             # failsafe; use "mps" if stable on your machine
)

detection_summary = run_detection_evaluation(detection_cfg)
detection_summary

In [ ]:
display(Image(filename=str(detection_cfg.output_dir / "plots" / "detection_metrics.png")))
display(Image(filename=str(detection_cfg.output_dir / "plots" / "detection_curves.png")))
print("Box overlays (green = ground truth, red = prediction):", detection_cfg.output_dir / "overlays")

## 5. Evaluation B - plot-status linear probe on frozen GAP features (LeJEPA vs COCO)
The backbone is frozen (including BatchNorm statistics); only a linear layer is trained on the concatenated P3/P4/P5 GAP vectors.

In [ ]:
probe_cfg = ProbeEvalConfig(
    image_dir=PLOT_STATUS_DIR,
    annotation_path=PLOT_STATUS_DIR / "annotations.json",
    output_dir=OUTPUT_ROOT / "plot_status_probe",
    weights=weights,
    label_mapping={"headland": 0, "between_plots": 1, "in_plot": 2},
    variants=("lejepa", "coco"),
    scales=("P3", "P4", "P5"),
    image_size=256,
    epochs=50,
    test_ratio=0.5,
)

probe_summary = run_probe_evaluation(probe_cfg)
probe_summary

In [ ]:
probe_plots = probe_cfg.output_dir / "plots"
display(Image(filename=str(probe_plots / "probe_metrics.png")))
display(Image(filename=str(probe_plots / "probe_loss_curves.png")))
for variant in probe_cfg.variants:
    display(Image(filename=str(probe_plots / f"confusion_{variant}.png")))
    display(Image(filename=str(probe_plots / f"pca2d_{variant}.png")))